# Leszek's setup: why his matched filter recovers more slowly and ours faster

Section 4 of the draft says the filter matched to the true noise ($\beta^* = 0.2$) recovers from a change
1.2 to 2.2 times more slowly than the Laplacian filters. Leszek's runs (`sims/sep18/ggbench.py`, results in
`ggbneg_summary_ar0.0.json`, overleaf commit `03e62fa`) give, for the recovery time matched / Laplacian:

| | vs exact | vs minorized |
|---|---|---|
| sKF | 1.27 | 1.22 |
| fKF | 2.14 | 2.22 |

Our runs of the same test give less than 1: notebook 09 (sKF) 0.79 / 0.81, `fkf.ipynb` (fKF) 0.90 / 0.94.

**The differences between the two setups**, from his script:

| | Leszek (`ggbench.py`) | notebook 09 |
|---|---|---|
| (i) input | white | AR($-0.9$) |
| (ii) change | at $T_c = 6000$, run $T = 12000$ | at $N = 96000$, run $2N$ |
| (iii) floor and tuning | mean of $[T_c - 1500, T_c)$ of the same run; parameter by bisection so that window is $-20$ dB | separate run at rest, mean of its last quarter; grid search |
| (iv) reading | 100-sample centred moving average before the 3 dB test | raw curve |
| (v) response, realisations | random, decaying, unit norm; $R = 40$, seed 1 | room of section 7.1; $R = 20$, seeds 0-19 |
| also | $v_0 = 1/M$; first update at $t = 0$ on a full window; misalignment after the update; test $\le$ | $v_0 = 2$; first update at $t = M$; misalignment before the update; test $<$ |

The filters are the same recursions: $\tilde v_t = v_{t-1} + \varepsilon$, $\sigma_t^2 = \tilde v_t\|\boldsymbol{x}_t\|^2$,
$v_t = \tilde v_t(1 - \chi_t'/M)$, $b_\eta = \mathrm{E}|\eta_t|$, and the fKF with $v$ fixed.

**Plan.**
1. **Anchor:** his signals and protocol, our filter code. It must give his numbers.
2. **Our setup** (notebook 09's, with its protocol), sKF and fKF, as the baseline.
3. **One change at a time** from our setup: (a) white input; (b) the change after 6000 samples.
4. If needed, the remaining factors one at a time, until the ratio is explained.

## 1. Imports

In [ ]:
import json
import time

import numpy as np
from matplotlib import pyplot as plt
from scipy.signal import lfilter
from scipy.special import gamma as G
from scipy.special import gammaln, log_ndtr
from scipy.stats import gennorm
import rir_generator as rir

%config InlineBackend.figure_format = 'svg'
NOTEBOOK_START = time.time()
LESZEK = "/Users/ramirouffelmann/Documents/INRS/paper2/overleaf/sims/sep18/"   # read only
print("imports ready")

## 2. The filters

Our code: the corrections of notebooks 07 and 08 (copied from notebook 09) and of `fkf.ipynb`, and the
batched sKF and fKF of `fkf.ipynb`, with two options for Leszek's protocol: the initial $v_0$ and the first
step that updates. The matched correction is notebook 08's quadrature written over arrays; `clipped-matched.ipynb`
checks it against the original to roundoff, and it is checked again here.

In [ ]:
M = 128
BETA = 0.2
SNR_DB = 5.0
VAR_THETA_0 = 2.0           # our v_0; Leszek's is 1/M
FS = 8000


# === IGNACIO: chi_laplacian and the quadrature - copied verbatim, NOT edited ===
# source: branch ignacio/joint-vs-marginal-vs-minorized, notebooks/09_matcheado.ipynb, commit 6e6dabc
# (chi_laplacian from notebook 07, f26ed36; quadrature from notebook 08, 0ed92a4)
def log_mills(z):
    """log R(z), with R(z) = Phi(-z)/phi(z) the Mills ratio, eq. (40). R grows like e^{z^2/2} for
    negative z and overflows, so it is only ever handled through its logarithm."""
    return log_ndtr(-z) + 0.5*z**2 + 0.5*np.log(2*np.pi)


def chi_laplacian(e, sigma, b_eta):
    """Correction chi_t(e_t) and its slope chi'_t(e_t) for Laplacian noise, eqs. (39), (41) and (43)."""
    u = e/sigma                                        # u_t = e_t / sigma_t
    k_t = sigma/b_eta                                  # k_t = sigma_t / b_eta
    tau = sigma**2/b_eta                               # tau_t = sigma_t^2 / b_eta, the largest correction
    log_R_minus = log_mills(k_t - u)                   # log R(k_t - u_t)
    log_R_plus = log_mills(k_t + u)                    # log R(k_t + u_t)
    Lambda = np.tanh((log_R_minus - log_R_plus)/2)     # (R- - R+)/(R- + R+), in (-1, 1)
    chi = tau*Lambda                                   # chi = tau Lambda
    chi_slope = (2*k_t*np.exp(-np.logaddexp(log_R_minus, log_R_plus))   # 2k / (R- + R+)
                 - k_t**2*(1 - Lambda**2))                               # - k^2 (1 - Lambda^2)
    return chi, chi_slope


L_WINDOW = 10               # the prior N(s; 0, sigma^2) is below e^{-50} beyond L_WINDOW sigma


def log_likelihood(u, density, params):
    """log p_eta(u) up to a constant: the noise density the filter assumes, in eqs. (21) and (24)."""
    if density == "gg":                                # generalized Gaussian, shape beta, scale alpha
        return -(np.abs(u)/params["alpha"])**params["beta"]
    # Student-t, nu degrees of freedom and scale c: -(nu + 1)/2 log(1 + (u/c)^2/nu)
    return -0.5*(params["nu"] + 1)*np.log1p((u/params["scale"])**2/params["nu"])


def piece_in_s(e, s_a, s_b, density, params):
    """Gauss-Legendre nodes on [s_a, s_b], directly in s. Returns the nodes and the log of
    (weight x length x likelihood p_eta(e_t - s)), the prior left out."""
    length = s_b - s_a                                 # length of the piece
    s = s_a + length*params["nodes"]                   # nodes mapped from [0, 1] to [s_a, s_b]
    log_q = params["log_weights"] + np.log(length) + log_likelihood(e - s, density, params)
    return s, log_q


def piece_in_z(e, s_a, s_b, params):
    """Gauss-Legendre nodes on [s_a, s_b] in the variable z = (|e_t - s|/alpha)^beta, where the
    cusp of the likelihood at s = e_t becomes the smooth e^{-z}. Same output as piece_in_s."""
    alpha = params["alpha"]
    beta = params["beta"]
    side = 1.0 if s_a >= e else -1.0                   # the piece lies above or below e_t
    z_a = (abs(e - s_a)/alpha)**beta                   # z at the two ends of the piece
    z_b = (abs(e - s_b)/alpha)**beta
    z_low = min(z_a, z_b)
    length = max(z_a, z_b) - z_low                     # length of the piece in z
    z = z_low + length*params["nodes"]                 # nodes mapped from [0, 1] to the piece
    log_z = np.log(z)
    s = e + side*alpha*np.exp(log_z/beta)              # s = e_t + side alpha z^(1/beta)
    # log of weight x length x ds/dz x likelihood, with ds/dz = (alpha/beta) z^(1/beta - 1)
    # and the likelihood e^{-z}
    log_q = params["log_weights"] + np.log(length*alpha/beta) + (1/beta - 1)*log_z - z
    return s, log_q


def chi_quadrature(e, sigma, density, params):
    """chi_t(e_t) and chi'_t(e_t) of eq. (30), from the mean and variance of the scalar
    posterior (21), by Gauss-Legendre quadrature. Same output as chi_laplacian."""
    # Cuts at the peak of the prior (0), the peak of the likelihood (e_t) and the ends of the
    # prior (+-L sigma), so that both peaks sit at the ends of pieces, where the nodes crowd.
    cuts = sorted([-L_WINDOW*sigma, 0.0, e, L_WINDOW*sigma])
    s_all = []                                         # nodes in s of every piece
    log_q_all = []                                     # log of their weights, eq. (21) times ds
    for piece in range(3):
        s_a = cuts[piece]
        s_b = cuts[piece + 1]
        if s_b <= s_a:                                 # e_t on a cut: this piece is empty
            continue
        if density == "gg" and params["beta"] < 1:
            s, log_q = piece_in_z(e, s_a, s_b, params)
        else:
            s, log_q = piece_in_s(e, s_a, s_b, density, params)
        s_all.append(s)
        log_q_all.append(log_q - s*s/(2*sigma**2))     # times the prior N(s; 0, sigma^2), eq. (21)
    s = np.concatenate(s_all)
    log_q = np.concatenate(log_q_all)
    q = np.exp(log_q - log_q.max())                    # largest weight becomes 1, nothing underflows
    total = q.sum()
    mean = (q @ s)/total                               # E[s_t | y_1:t]
    var = (q @ (s - mean)**2)/total                    # Var[s_t | y_1:t]
    return mean, 1 - var/sigma**2                      # chi and chi', eq. (30)


def density_params(density, shape, scale, n_nodes):
    """Everything chi_quadrature needs about one density: shape, scale and the nodes."""
    nodes, weights = np.polynomial.legendre.leggauss(n_nodes)   # on [-1, 1], computed once
    nodes = (nodes + 1)/2                              # moved to [0, 1]: a piece [a, b] takes
    weights = weights/2                                # a + (b - a) node, weight (b - a) weight
    if density == "gg":
        return {"beta": shape, "alpha": scale, "nodes": nodes, "log_weights": np.log(weights)}
    return {"nu": shape, "scale": scale, "nodes": nodes, "log_weights": np.log(weights)}
# === end of the copied block ===


# Copied from fkf.ipynb, commit 058a9f9.
def chi_minorized(e, sigma, b_eta):
    """Eq. (45) of the draft (chi.minorized): the minorized correction tau e/(tau + |e|), and the
    ratio chi/e that replaces chi' in the variance update (Table 2). The fKF uses the first only."""
    tau = sigma**2/b_eta                               # tau_t = sigma_t^2 / b_eta
    ratio = tau/(tau + np.abs(e))                      # chi_min / e, in (0, 1]
    return ratio*e, ratio


def vectorised_gg(params):
    """chi_quadrature of notebook 08 for the generalized Gaussian with beta < 1, over arrays of
    errors at once: the same three pieces, the same change of variable (piece_in_z), the same nodes
    and weights. New here; checked against chi_quadrature below. b_eta is not used."""
    alpha, beta = params["alpha"], params["beta"]
    nodes, log_weights = params["nodes"], params["log_weights"]

    def chi_fn(e, sigma, b_eta=None):
        e = np.asarray(e, dtype=float)
        sigma = np.broadcast_to(np.asarray(sigma, dtype=float), e.shape)
        # cuts at -L sigma, 0, e_t and +L sigma, sorted: three pieces per error
        cuts = np.sort(np.stack([-L_WINDOW*sigma, np.zeros_like(e), e, L_WINDOW*sigma], axis=-1), axis=-1)
        s_a, s_b = cuts[..., :3], cuts[..., 1:]
        e_col = e[..., None]
        side = np.where(s_a >= e_col, 1.0, -1.0)      # the piece lies above or below e_t
        z_a = (np.abs(e_col - s_a)/alpha)**beta        # z = (|e_t - s|/alpha)^beta at both ends
        z_b = (np.abs(e_col - s_b)/alpha)**beta
        z_low = np.minimum(z_a, z_b)
        length = np.maximum(z_a, z_b) - z_low
        with np.errstate(divide="ignore", invalid="ignore"):   # empty pieces, masked just below
            z = z_low[..., None] + length[..., None]*nodes
            log_z = np.log(z)
            s = e_col[..., None] + side[..., None]*alpha*np.exp(log_z/beta)
            log_q = (log_weights + np.log(length*alpha/beta)[..., None] + (1/beta - 1)*log_z - z
                     - s*s/(2*sigma[..., None, None]**2))     # times the prior N(s; 0, sigma^2)
        full = (s_b > s_a)[..., None]                  # e_t on a cut: that piece is empty
        log_q = np.where(full, log_q, -np.inf).reshape(e.shape + (-1,))
        s = np.where(full, s, 0.0).reshape(e.shape + (-1,))
        q = np.exp(log_q - log_q.max(axis=-1, keepdims=True))
        total = q.sum(axis=-1)
        mean = (q*s).sum(axis=-1)/total                # E[s_t | y_1:t]
        var = (q*(s - mean[..., None])**2).sum(axis=-1)/total   # Var[s_t | y_1:t]
        return mean, 1 - var/sigma**2                  # chi and chi', eq. (30)
    return chi_fn


# sKF_batch and fKF_batch: copied from fkf.ipynb, commit 058a9f9 (sKF_batch from vkf-kf.ipynb,
# commit 2724d95), with options added and nothing else changed. Defaults reproduce the originals.
#   flip_at: misalignment against -h from that step on (fkf.ipynb had it for the fKF only)
#   clip:    chi' -> max(chi', 0) in the variance update, eq. (36); the mean update is untouched
#   record:  also return chi'_t and v_t at every step
#   v0:      initial v (VAR_THETA_0 = 2 in every notebook of ours; 1/M in Leszek's ggbench.py)
#   wait:    first step that updates (M: wait for a full window, as every notebook of ours)
def _roll_in(X_t, x_win):
    x_win = np.roll(x_win, 1, axis=1)
    x_win[:, 0] = X_t
    return x_win


def sKF_batch(X, D, h, b_eta, epsilon, chi_fn, flip_at=None, clip=False, record=False,
              v0=VAR_THETA_0, wait=None):
    L, (B, n) = len(h), X.shape
    first = L if wait is None else wait
    w, x_t = np.zeros((B, L)), np.zeros((B, L))
    v = np.full(B, float(v0))
    eps = np.asarray(epsilon, dtype=float)
    mis = np.empty((B, n))
    if record:
        slope_hist, v_hist = np.full((B, n), np.nan), np.full((B, n), np.nan)
    for t in range(n):
        x_t = _roll_in(X[:, t], x_t)
        e = D[:, t] - np.einsum("bm,bm->b", x_t, w)
        dw = (w - h) if flip_at is None or t < flip_at else (w + h)
        mis[:, t] = np.einsum("bm,bm->b", dw, dw)
        if t < first:
            continue
        v_tilde = v + eps
        power = np.einsum("bm,bm->b", x_t, x_t)
        sigma = np.sqrt(v_tilde*power)
        chi, slope = chi_fn(e, sigma, b_eta)
        w = w + x_t*(chi/power)[:, None]
        v = v_tilde*(1 - (np.maximum(slope, 0.0) if clip else slope)/L)
        if record:
            slope_hist[:, t], v_hist[:, t] = slope, v
    if record:
        return mis, slope_hist, v_hist
    return mis


def fKF_batch(X, D, h, b_eta, v, chi_fn, flip_at=None, wait=None):
    """B fKFs of eq. (37) side by side. Misalignment against h, or against -h from step flip_at on."""
    L, (B, n) = len(h), X.shape
    first = L if wait is None else wait
    w, x_t = np.zeros((B, L)), np.zeros((B, L))
    v = np.asarray(v, dtype=float)
    mis = np.empty((B, n))
    for t in range(n):
        x_t = _roll_in(X[:, t], x_t)
        e = D[:, t] - np.einsum("bm,bm->b", x_t, w)
        dw = (w - h) if flip_at is None or t < flip_at else (w + h)
        mis[:, t] = np.einsum("bm,bm->b", dw, dw)
        if t < first:
            continue
        power = np.einsum("bm,bm->b", x_t, x_t)
        chi, _ = chi_fn(e, np.sqrt(v*power), b_eta)
        w = w + x_t*(chi/power)[:, None]
    return mis


def correction(likelihood, noise_scale):
    """chi for one of the three likelihoods; the matched one takes the true generalized Gaussian scale."""
    if likelihood == "minorized":
        return chi_minorized
    if likelihood == "exact":
        return chi_laplacian
    return vectorised_gg(density_params("gg", BETA, noise_scale, 100))


FILTERS = [(family, likelihood) for family in ("sKF", "fKF") for likelihood in ("minorized", "exact", "matched")]

# the vectorised quadrature against chi_quadrature, on a small set
params_check = density_params("gg", BETA, 8.435e-06, 100)
rng_check = np.random.default_rng(0)
sig_check = 10**rng_check.uniform(-3, 0.5, 2000)
e_check = sig_check*rng_check.standard_normal(2000)*10**rng_check.uniform(-3, 2, 2000)
ref = np.array([chi_quadrature(a, b, "gg", params_check) for a, b in zip(e_check, sig_check)])
got = vectorised_gg(params_check)(e_check, sig_check)
print(f"vectorised quadrature against chi_quadrature, 2000 errors: max |chi diff|/sigma "
      f"{np.max(np.abs(got[0] - ref[:, 0])/sig_check):.1e}, max |chi' diff| {np.max(np.abs(got[1] - ref[:, 1])):.1e}")

## 3. The anchor: Leszek's signals and protocol, our filters

His signals, drawn exactly as `ggbench.py` draws them (`default_rng(1)`, the same calls in the same order),
with `AR = 0`: $\boldsymbol{h}_1$ random with a decay of $T_{60} = 0.2$ s at 8 kHz, unit norm,
$\boldsymbol{h}_2 = -\boldsymbol{h}_1$; unit-power white input; noise $\eta = \mathrm{sign}\cdot\alpha_*\,\Gamma(1/\beta^*)^{1/\beta^*}$.

His loop, `run`, is our recursion with $v_0 = 1/M$ and an update from $t = 0$, since his regressor at $t$ is
already a full window, `x[t:t+M][::-1]`. Our batched filters take the input one sample at a time, so they
are fed his $x$ from the start and his $d_t$ at step $t' = t + M - 1$, and told to update from $t' = M - 1$.
He records the misalignment after the update, which is ours at $t' + 1$.

His `tune`, `floor_db`, `smooth` and `times` are rewritten below line for line from `ggbench.py`.

**Checks:** each filter's parameter, floor, convergence and recovery against his JSON; and our curve at his
parameter against the curve he saved (`ggbneg_<filter>_ar0.0.npy`).

In [ ]:
# Written from Leszek's sims/sep18/ggbench.py (overleaf commit 03e62fa), AR = 0, line for line.
R_L, TC_L, T_L = 40, 6000, 12000
rng0 = np.random.default_rng(1)
h1 = rng0.standard_normal(M)*np.exp(-6.9*np.arange(M)/(0.2*8000))
h1 = h1/np.linalg.norm(h1)
x_l = rng0.standard_normal((R_L, T_L + M))       # AR = 0: his AR loop leaves x = u, and divides by 1
veta_l = 10**(-SNR_DB/10)                        # E[(x'h)^2] = ||h||^2 = 1
alpha_l = np.sqrt(veta_l*G(1/BETA)/G(3/BETA))    # gg_alpha of his ggq.py
eta_l = np.sign(rng0.standard_normal((R_L, T_L)))*alpha_l*rng0.gamma(1/BETA, 1, (R_L, T_L))**(1/BETA)
b_l = np.sqrt(veta_l)*G(2/BETA)/np.sqrt(G(1/BETA)*G(3/BETA))    # E|eta|

# his d_t = x[t:t+M][::-1] h + eta_t, with h -> -h from TC_L on
y_l = np.stack([np.convolve(h1, x_l[r])[M - 1:M - 1 + T_L] for r in range(R_L)])
y_l[:, TC_L:] = -y_l[:, TC_L:]
X_l = x_l                                                               # T_L + M samples
D_l = np.concatenate([np.zeros((R_L, M - 1)), y_l + eta_l, np.zeros((R_L, 1))], axis=1)


def run_leszek(family, likelihood, par):
    """His run(kind, par), through our batched filters. Returns his mis[t], t = 0..T_L-1."""
    chi = correction(likelihood, alpha_l)
    if family == "sKF":
        mis = sKF_batch(X_l, D_l, h1, b_l, np.full(R_L, par), chi, flip_at=TC_L + M, v0=1.0/M, wait=M - 1)
    else:
        mis = fKF_batch(X_l, D_l, h1, b_l, np.full(R_L, par), chi, flip_at=TC_L + M, wait=M - 1)
    return mis.mean(axis=0)[M:M + T_L]


def smooth(m, k=100):
    return np.convolve(m, np.ones(k)/k, mode="same")


def floor_db(m, tc=TC_L):
    return 10*np.log10(np.mean(m[tc - 1500:tc]))


def times(m, fl, tc=TC_L):
    d = 10*np.log10(smooth(m))
    thr = fl + 3
    i1 = np.argmax(d[:tc] <= thr) if np.any(d[:tc] <= thr) else np.nan
    j = d[tc:]
    i2 = np.argmax(j <= thr) if np.any(j <= thr) else np.nan
    return i1, i2


def tune(run, lo, hi, target=-20.0, it=14):
    """Bisection in log(par): floor increases with par."""
    for _ in range(it):
        mid = np.sqrt(lo*hi)
        if floor_db(run(mid)) > target:
            hi = mid
        else:
            lo = mid
    p = np.sqrt(lo*hi)
    return p, run(p)


KIND = {("fKF", "minorized"): "fLm", ("fKF", "exact"): "fLe", ("fKF", "matched"): "fGG",
        ("sKF", "minorized"): "sLm", ("sKF", "exact"): "sLe", ("sKF", "matched"): "sGG"}
BRACKET = {"fKF": (1e-7, 1e-1), "sKF": (1e-12, 1e-3)}          # his lo, hi
his = json.load(open(LESZEK + "ggbneg_summary_ar0.0.json"))

anchor = {}
print(f"{'filter':<16}{'par':>11}{'his':>11}{'floor':>8}{'conv':>6}{'his':>6}{'rec':>6}{'his':>6}"
      f"{'curve at his par, max |dB diff|':>33}")
for family, likelihood in FILTERS:
    start = time.time()
    run = lambda par: run_leszek(family, likelihood, par)
    p, m = tune(run, *BRACKET[family])
    fl = floor_db(m)
    conv, rec = times(m, fl)
    k = KIND[(family, likelihood)]
    saved = np.load(LESZEK + f"ggbneg_{k}_ar0.0.npy")
    diff = np.max(np.abs(10*np.log10(run(his[k]["par"])) - 10*np.log10(saved)))
    anchor[(family, likelihood)] = dict(value=p, floor=fl, conv=conv, rec=rec, curve=m)
    print(f"{family + ', ' + likelihood:<16}{p:>11.4e}{his[k]['par']:>11.4e}{fl:>8.2f}{conv:>6.0f}"
          f"{his[k]['t_conv']:>6.0f}{rec:>6.0f}{his[k]['t_rec']:>6.0f}{diff:>33.1e}   [{time.time() - start:.0f} s]")

## 4. Our setup, with notebook 09's protocol

Notebook 09's scenario and protocol, as in `clipped-matched.ipynb`: grid search at rest ($R = 20$,
$N = 96000$, settled points only, interpolated to $-20$ dB), then $2N$ steps with the flip at $N$; floor =
the run at rest, recovered = first step after the change within 3 dB of it, on the raw curve. The sKF
sweeps $\varepsilon$ over logspace$(-8, -3, 11)$ (notebook 09), the fKF $v$ over logspace$(-6, -1, 11)$
(`fkf.ipynb`). The input colour is a parameter, so the same code runs (a).

The sKF rows must repeat notebook 09 (4357 / 4756 / 3630 steps at rest, 7340 / 7584 / 5979 to recover).
The fKF rows use notebook 09's protocol here, not `fkf.ipynb`'s shorter search, so they can differ from it
by a little.

In [ ]:
N = 96000
R = 20
WARMUP = 500
TARGET_DB = -20.0
DRIFT_DB = 0.5
EPS_GRID = np.logspace(-8, -3, 11)
V_GRID = np.logspace(-6, -1, 11)
ROOM, T60, C_SOUND, SRC, MIC = [5, 10, 6], 0.2, 340, [1, 2.5, 2], [1, 1.5, 1]
ho = rir.generate(c=C_SOUND, fs=FS, r=MIC, s=SRC, L=ROOM, reverberation_time=T60, nsample=M).flatten()
ho = ho/np.linalg.norm(ho)
lags = np.abs(np.subtract.outer(np.arange(M), np.arange(M)))


def noise_of(ar_a):
    """Noise scale and b_eta = E|eta| at 5 dB for this input: notebook 09's lines, AR as a parameter."""
    P_signal = ho @ (ar_a**lags if ar_a != 0 else np.eye(M)) @ ho
    var_eta = P_signal/10**(SNR_DB/10)
    scale_gg = np.sqrt(var_eta/np.exp(gammaln(3/BETA) - gammaln(1/BETA)))
    return scale_gg, scale_gg*np.exp(gammaln(2/BETA) - gammaln(1/BETA))


def signals(seed, ar_a, n, change_at=None):
    """generate_signals / generate_signals_change of notebook 09 (commit 6e6dabc), with the AR
    coefficient, the length and the instant of the flip as parameters. Same draws, same order."""
    scale_gg, _ = noise_of(ar_a)
    rng = np.random.default_rng(seed)
    u = np.sqrt(1 - ar_a**2)*rng.standard_normal(n + WARMUP)
    x = lfilter([1.0], [1.0, -ar_a], u)[WARMUP:]
    y = np.convolve(ho, x)[:n]
    if change_at is not None:
        y[change_at:] = -y[change_at:]
    eta = gennorm.rvs(BETA, scale=scale_gg, size=n, random_state=rng)
    return x, y + eta


def stack(sigs):
    return np.array([s[0] for s in sigs]), np.array([s[1] for s in sigs])


def filter_fn(family, likelihood, ar_a):
    scale_gg, b_eta = noise_of(ar_a)
    chi = correction(likelihood, scale_gg)
    if family == "sKF":
        return lambda X, D, p, **kw: sKF_batch(X, D, ho, b_eta, p, chi, **kw)
    return lambda X, D, p, **kw: fKF_batch(X, D, ho, b_eta, p, chi, **kw)


# === IGNACIO: steady_state and is_settled - copied verbatim, NOT edited ===
# source: branch ignacio/joint-vs-marginal-vs-minorized, notebooks/09_matcheado.ipynb, commit 6e6dabc
def steady_state(misalignment):
    """Floor in dB, and the first step within 3 dB of it."""
    tail = slice(3*len(misalignment)//4, len(misalignment))
    floor = 10*np.log10(misalignment[tail].mean())
    db = 10*np.log10(misalignment)
    reached = int(np.argmax(db < floor + 3))
    return floor, reached


def is_settled(misalignment, drift):
    """Settled within the run: within 3 dB of the floor by half of it, and no longer descending."""
    n = len(misalignment)
    floor, reached = steady_state(misalignment)
    third_quarter = 10*np.log10(misalignment[n//2:3*n//4].mean())
    # reached = 0: the run never left the 3 dB band around its start, so it never converged either
    return bool(0 < reached <= n/2 and third_quarter - floor <= drift)
# === end of the copied block ===



def notebook09_protocol(ar_a):
    """Grid at rest, then the change run; floor from the run at rest."""
    X, D = stack([signals(s, ar_a, N) for s in range(R)])
    Xc, Dc = stack([signals(s, ar_a, 2*N, N) for s in range(R)])
    out = {}
    for family, likelihood in FILTERS:
        start = time.time()
        fn = filter_fn(family, likelihood, ar_a)
        grid = EPS_GRID if family == "sKF" else V_GRID
        g = len(grid)
        curves = fn(np.repeat(X, g, axis=0), np.repeat(D, g, axis=0), np.tile(grid, R)).reshape(R, g, -1).mean(axis=0)
        floors = np.array([steady_state(c)[0] for c in curves])
        settled = np.array([is_settled(c, DRIFT_DB) for c in curves])
        values, kept = grid[settled], floors[settled]
        if TARGET_DB < kept.min():
            value, status = values[np.argmin(kept)], "target not reached"
        else:
            order = np.argsort(kept)
            value, status = 10**np.interp(TARGET_DB, kept[order], np.log10(values)[order]), "ok"
        floor, steps = steady_state(fn(X, D, np.full(R, value)).mean(axis=0))
        curve = fn(Xc, Dc, np.full(R, value), flip_at=N).mean(axis=0)
        back = 10*np.log10(curve[N:]) < floor + 3
        out[(family, likelihood)] = dict(value=value, status=status, floor=floor, steps=steps, curve=curve,
                                         rec=int(np.argmax(back)) if back.any() else -1)
        r = out[(family, likelihood)]
        print(f"   {family + ', ' + likelihood:<16}{value:>11.3e}  {status:<4}{floor:>8.2f}{steps:>7d}{r['rec']:>7d}"
              f"   [{time.time() - start:.0f} s]")
    return out


def show_ratios(out, label):
    text = []
    for family in ("sKF", "fKF"):
        m = out[(family, "matched")]["rec"]
        text.append(f"{family}: matched/exact {m/out[(family, 'exact')]['rec']:.2f}, "
                    f"matched/minorized {m/out[(family, 'minorized')]['rec']:.2f}")
    print(f"{label}:  " + ";   ".join(text))


print(f"   {'filter':<16}{'eps or v':>11}  {'':<4}{'floor':>8}{'steps':>7}{'rec':>7}")
base = notebook09_protocol(-0.9)
show_ratios(base, "our setup")

## 5. (a) White input, nothing else changed

The same code with the AR coefficient at 0. Signal power 1, so the noise and $b_\eta$ follow. Same room
response, seeds, grids and protocol.

In [ ]:
print(f"   {'filter':<16}{'eps or v':>11}  {'':<4}{'floor':>8}{'steps':>7}{'rec':>7}")
white = notebook09_protocol(0.0)
show_ratios(white, "(a) white input")

## 6. (b) The change after 6000 samples

**The protocol of (iii) is out of reach here.** With AR($-0.9$) input the filters need about 4000-5000
steps to converge (section 4), so at $T_c = 6000$ the window $[4500, 6000)$ is still partly transient.
Scanning the parameter shows how deep that window can go at all: the level is U-shaped in the parameter
(too small: not converged yet; too large: a high floor), and its bottom is above $-20$ dB for five of the
six filters. Leszek's bisection assumes the level rises with the parameter, so at $-20$ dB it would slide to
the low end of the bracket and return a filter that has not converged.

In [ ]:
TC = 6000
Xb, Db = stack([signals(s, -0.9, TC + N, TC) for s in range(R)])
SCAN = {"sKF": np.logspace(-12, -3, 19), "fKF": np.logspace(-7, -1, 13)}
deepest = {}
print(f"{'filter':<16}{'deepest window [dB]':>21}{'at':>10}")
for family, likelihood in FILTERS:
    fn = filter_fn(family, likelihood, -0.9)
    grid = SCAN[family]
    g = len(grid)
    curves = fn(np.repeat(Xb[:, :TC], g, axis=0), np.repeat(Db[:, :TC], g, axis=0),
                np.tile(grid, R)).reshape(R, g, -1).mean(axis=0)
    level = np.array([floor_db(c, TC) for c in curves])
    deepest[(family, likelihood)] = grid[np.argmin(level)]
    print(f"{family + ', ' + likelihood:<16}{level.min():>21.2f}{grid[np.argmin(level)]:>10.1e}")
print("window: mean misalignment over [4500, 6000), R = 20, AR(-0.9)")

So (b) is run in two forms, each labelled with what "floor" means in it:

- **(b1)** Only the change time moves. Each filter keeps its parameter and its floor at rest from section 4
  (a separate run, mean of its last quarter), and recovery is read against that floor. At $T_c = 6000$ the
  filters have not all reached that floor before the change; the table prints how far they are.
- **(b2)** The change at 6000 with Leszek's floor and tuning, at the deepest level all six reach: $-15$ dB.
  Floor = mean of $[4500, 6000)$ of the same run, parameter by his 14-step bisection, started on the rising
  side of the U (from the scan's deepest point up to his upper bracket).

Both read the raw curve, as notebook 09.

In [ ]:
b1 = {}
print("(b1)")
print(f"   {'filter':<16}{'floor at rest':>14}{'window before change':>22}{'rec':>7}")
for family, likelihood in FILTERS:
    r = base[(family, likelihood)]
    curve = filter_fn(family, likelihood, -0.9)(Xb, Db, np.full(R, r["value"]), flip_at=TC).mean(axis=0)
    back = 10*np.log10(curve[TC:]) < r["floor"] + 3
    b1[(family, likelihood)] = dict(rec=int(np.argmax(back)) if back.any() else -1, curve=curve)
    print(f"   {family + ', ' + likelihood:<16}{r['floor']:>14.2f}{floor_db(curve, TC):>22.2f}"
          f"{b1[(family, likelihood)]['rec']:>7d}")
show_ratios(b1, "(b1) change at 6000")


def window_protocol(X, D, ar_a, tc, target, smoothing, bracket):
    """Leszek's floor and tuning on our setup: window floor, bisection, recovery on the raw or the
    smoothed curve. Tuning runs need only the first tc steps."""
    out = {}
    for family, likelihood in FILTERS:
        fn = filter_fn(family, likelihood, ar_a)
        run_rest = lambda p: fn(X[:, :tc], D[:, :tc], np.full(len(X), p)).mean(axis=0)
        lo, hi = bracket(family, likelihood)
        for _ in range(14):
            mid = np.sqrt(lo*hi)
            if floor_db(run_rest(mid), tc) > target:
                hi = mid
            else:
                lo = mid
        p = np.sqrt(lo*hi)
        curve = fn(X, D, np.full(len(X), p), flip_at=tc).mean(axis=0)
        fl = floor_db(curve, tc)
        d = 10*np.log10(smooth(curve) if smoothing else curve)
        back = d[tc:] < fl + 3
        out[(family, likelihood)] = dict(value=p, floor=fl, rec=int(np.argmax(back)) if back.any() else -1,
                                         curve=curve)
        print(f"   {family + ', ' + likelihood:<16}{p:>11.3e}{fl:>8.2f}{out[(family, likelihood)]['rec']:>7d}")
    return out


print("\n(b2)")
print(f"   {'filter':<16}{'eps or v':>11}{'window':>8}{'rec':>7}")
UPPER = {"sKF": 1e-3, "fKF": 1e-1}
b2 = window_protocol(Xb, Db, -0.9, TC, -15.0, False, lambda f, l: (deepest[(f, l)], UPPER[f]))
show_ratios(b2, "(b2) change at 6000, window floor at -15 dB")

## 7. From (a) toward Leszek's setup

(a) moves the ratio above 1 on its own (section 5). To see whether the rest of his protocol adds anything,
the factors are added on top of the white input, cumulatively: the change at 6000 with his window floor and
bisection at $-20$ dB (reachable with white input, which converges in about 1500 steps), then the
100-sample moving average. What still differs from the anchor after that is the response (random, $R = 40$,
seed 1), $v_0 = 1/M$ and the first update at $t = 0$.

In [ ]:
Xw, Dw = stack([signals(s, 0.0, TC + N, TC) for s in range(R)])
print("white input + change at 6000 + window floor and bisection at -20 dB")
print(f"   {'filter':<16}{'eps or v':>11}{'window':>8}{'rec':>7}")
white_tc = window_protocol(Xw, Dw, 0.0, TC, -20.0, False, lambda f, l: BRACKET[f])
show_ratios(white_tc, "white + Tc 6000 + window floor")
print("\nthe same, recovery read on the 100-sample moving average")
print(f"   {'filter':<16}{'eps or v':>11}{'window':>8}{'rec':>7}")
white_tc_smooth = window_protocol(Xw, Dw, 0.0, TC, -20.0, True, lambda f, l: BRACKET[f])
show_ratios(white_tc_smooth, "+ smoothing")

## 8. Summary

In [ ]:
def ratio_pair(out, family):
    m = out[(family, "matched")]["rec"]
    return m/out[(family, "exact")]["rec"], m/out[(family, "minorized")]["rec"]


json_rec = {key: his[k]["t_rec"] for key, k in KIND.items()}
json_out = {key: {"rec": value} for key, value in json_rec.items()}
ROWS = [("Leszek, his JSON (ggbneg_summary_ar0.0.json)", json_out),
        ("anchor: his signals and protocol, our filters", anchor),
        ("our setup (notebook 09's, its protocol)", base),
        ("(a) white input only", white),
        ("(b1) change at 6000 only, floor at rest", b1),
        ("(b2) change at 6000, window floor, -15 dB", b2),
        ("(a) + change at 6000 + window floor, -20 dB", white_tc),
        ("   + 100-sample moving average", white_tc_smooth)]
print(f"{'setup':<48}{'sKF m/exact':>12}{'sKF m/min':>11}{'fKF m/exact':>13}{'fKF m/min':>11}")
for label, out in ROWS:
    s = ratio_pair(out, "sKF")
    f = ratio_pair(out, "fKF")
    print(f"{label:<48}{s[0]:>12.2f}{s[1]:>11.2f}{f[0]:>13.2f}{f[1]:>11.2f}")
print("recovery time, matched over Laplacian (exact or minorized); above 1, the matched filter is slower")

### How the input changes the recovery

The runs of sections 4 and 5, read at several levels after the change, as in `fkf.ipynb`: the first step
below each level.

In [ ]:
def time_to(curve, level_db, start=N):
    below = 10*np.log10(curve[start:]) < level_db
    return int(np.argmax(below)) if below.any() else -1


for label, out in (("our setup, AR(-0.9)", base), ("(a) white input", white)):
    print(label)
    print(f"   {'':<5}{'level [dB]':>11}{'minorized':>11}{'exact':>8}{'matched':>9}{'m/exact':>9}{'m/min':>7}")
    for family in ("sKF", "fKF"):
        for level in (0.0, -5.0, -10.0, -15.0, "floor + 3"):
            steps = {l: (out[(family, l)]["rec"] if level == "floor + 3" else time_to(out[(family, l)]["curve"], level))
                     for l in ("minorized", "exact", "matched")}
            text = f"{level:>11}" if isinstance(level, str) else f"{level:>11.0f}"
            print(f"   {family:<5}{text}{steps['minorized']:>11d}{steps['exact']:>8d}{steps['matched']:>9d}"
                  f"{steps['matched']/steps['exact']:>9.2f}{steps['matched']/steps['minorized']:>7.2f}")

**Figure 1.** Misalignment across the change, time counted from it: top, the sKF; bottom, the fKF; left,
our setup (AR($-0.9$)); right, (a), the same with white input. Dotted, in each curve's colour: the instant it
gets back within 3 dB of its floor at rest. Black, dashed: the change. The time axes differ: white input
recovers about five times faster.

In [ ]:
COLOUR = {"minorized": "#2a78d6", "exact": "#eb6834", "matched": "#1baf7a"}   # as fkf.ipynb
t_change = (np.arange(2*N) - N)/FS
fig, axes = plt.subplots(2, 2, figsize=(12, 7.5), constrained_layout=True)
for col, (title, out) in enumerate((("our setup, AR(-0.9)", base), ("(a) white input", white))):
    slowest = max(r["rec"] for r in out.values())
    keep = (t_change >= -0.1*slowest/FS) & (t_change <= 1.5*slowest/FS)
    for row, family in enumerate(("sKF", "fKF")):
        ax = axes[row, col]
        for likelihood in ("minorized", "exact", "matched"):
            r = out[(family, likelihood)]
            ax.plot(t_change[keep], 10*np.log10(r["curve"][keep]), color=COLOUR[likelihood], lw=1.4,
                    label=f"{likelihood}, back in {r['rec']/FS:.2f} s")
            ax.axvline(r["rec"]/FS, color=COLOUR[likelihood], ls=":", lw=1.1)
        ax.axvline(0, color="k", ls="--", lw=0.9)
        ax.set(ylim=(-24, 8), title=f"{family}, {title}", ylabel="misalignment [dB]")
        ax.grid(alpha=0.25)
        ax.legend(fontsize=8, loc="upper right")
for ax in axes[-1]:
    ax.set_xlabel("time from the change [s]")
fig.suptitle(r"Across $h_o \to -h_o$, $\beta^* = 0.2$, 5 dB, $b_\eta = \mathrm{E}|\eta|$, notebook 09's protocol")
plt.show()

## 9. Findings

**Anchor.** On Leszek's signals and protocol our filter code gives his numbers: every parameter to four
digits, and every convergence and recovery time to the step (sKF 730 / 758 / 650 and 1326 / 1275 / 1614;
fKF 959 / 948 / 1224 and 1565 / 1625 / 3476). At his parameters, our curves match the ones he saved to
1e-14 dB where he uses a closed form, and to 0.002 dB where he uses his trapezoid quadrature (both matched
filters, and his exact sKF, which takes its moments by quadrature at $\beta = 1$). So the filters are the same; only the setup differs.

**The input colour is the whole difference.** Changing only the input to white, in our setup and with our
protocol, gives matched / exact 1.30 (sKF) and 2.10 (fKF), against his 1.27 and 2.14. The change after
6000 samples does not move the ratio above 1, in either form ((b1): 0.82 and 0.93; (b2): 0.86 and 0.87).
On top of the white input, his window floor, his bisection and his moving average change the ratio by at
most 0.05 (sKF 1.27 / 1.21, fKF 2.10 / 2.20, against his 1.27 / 1.22 and 2.14 / 2.22). The random response,
$R = 40$ and $v_0 = 1/M$ are not needed to explain it.

| setup | sKF, m/exact | sKF, m/min | fKF, m/exact | fKF, m/min |
|---|---|---|---|---|
| Leszek (JSON) | 1.27 | 1.22 | 2.14 | 2.22 |
| anchor (his setup, our code) | 1.27 | 1.22 | 2.14 | 2.22 |
| our setup | 0.79 | 0.81 | 0.92 | 0.96 |
| (a) white input only | **1.30** | **1.26** | **2.10** | **2.22** |
| (b1) change at 6000 only | 0.82 | 0.83 | 0.93 | 0.96 |
| (b2) change at 6000, window floor at $-15$ dB | 0.86 | 0.84 | 0.87 | 0.86 |
| (a) + change at 6000 + window floor | 1.27 | 1.21 | 2.10 | 2.20 |
| + moving average | 1.27 | 1.21 | 2.10 | 2.20 |

**Why the input matters.** In both setups the matched filter starts the recovery late: at 0 dB it is behind
(sKF 1.09x the exact under AR input, 1.80x under white input; fKF 1.46x and 3.20x). It reads the first large
errors after the change as outliers. What differs is what comes next. Under AR($-0.9$) the recovery is long
(6000-8000 steps) and the matched filters descend faster near the floor, so they catch up and finish first.
Under white input the whole recovery takes 1200-1600 steps: from 0 dB to the floor the matched sKF takes 749
steps, the same as the exact one's 755, so the late start is never won back. The fKF starts later still
(3.20x at 0 dB), and its final ratio is the larger one.

So the draft's 1.2 to 2.2 holds for white input and does not hold for the coloured input of our setup.
Which input the paper should use is a question for Leszek; either way, the draft should say which one the
numbers are for.

In [ ]:
print(f"whole notebook: {(time.time() - NOTEBOOK_START)/60:.1f} min")